In [2]:
import torch
import timm
import numpy as np

from torchvision import transforms
from torch.utils.data import DataLoader

from wildlife_datasets.datasets import Lynx
from wildlife_tools.data import WildlifeDataset

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [4]:
model_name = "hf-hub:BVRA/MegaDescriptor-T-224"

model = timm.create_model(
    model_name,
    pretrained=True,
    num_classes=0
)

model = model.to(device)
model.eval()

print("Model loaded")

Model loaded


In [5]:
print(model.num_features)

768


In [6]:
dataset_version = ""

metadata = Lynx(f"data_rysy/rys_trening_data_Beno{dataset_version}")

print("Images:", len(metadata.df))
metadata.df.head()

Images: 319


,image_id,identity,path,date
0,0,Adam,rys_trening_data_Beno\Adam\Adam_1.JPG,2015-01-01
1,1,Adam,rys_trening_data_Beno\Adam\Adam_2.JPG,2015-01-01
2,2,Adam,rys_trening_data_Beno\Adam\Adam_3.JPG,2015-01-01
3,3,Adam,rys_trening_data_Beno\Adam\Adam_4.JPG,2015-01-01
4,4,Adam,rys_trening_data_Beno\Adam\Adam_5.JPG,2015-01-01


In [7]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

In [8]:
dataset = WildlifeDataset(
    metadata.df,
    metadata.root,
    transform=transform
)

print("Dataset size:", len(dataset))

Dataset size: 319


In [9]:
loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

In [10]:
model.eval()

all_embeddings = []

with torch.no_grad():

    for imgs, _ in loader:

        imgs = imgs.to(device)

        emb = model(imgs)

        all_embeddings.append(emb.cpu())

embeddings = torch.cat(all_embeddings).numpy()

print("Embeddings shape:", embeddings.shape)

Embeddings shape: (319, 768)
